# SleepWise - Exploratory Data Analysis
Run the cells top to bottom (**Kernel → Restart & Run All**). The notebook uses the same code as `train.py`
(`src/sleepwise/eda.py`), so the figures here match the ones the app shows.

> Everything below describes *associations in this dataset*. Correlation does not prove causation, and the data
> is not clinically validated.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from sleepwise import config, eda
from sleepwise.data_loader import load_raw_csv, validate_dataset, build_schema_report
from sleepwise.preprocessing import clean_dataframe, get_feature_roles

## 1. Load and validate
If this fails, the error message says where to place the CSV (`data/raw/Sleep_health_and_lifestyle_dataset.csv`).

In [ ]:
raw = load_raw_csv()
validate_dataset(raw)
print("Rows x columns:", raw.shape)
raw.head()

## 2. Shape, data types, missing values, duplicates

In [ ]:
display(raw.dtypes.to_frame("dtype"))
print("Missing values per column:"); display(raw.isna().sum().to_frame("missing"))
print("Exact duplicate rows            :", raw.duplicated().sum())
print("Duplicates when Person ID ignored:", raw.drop(columns=["Person ID"]).duplicated().sum())
print("Distinct rows when Person ID ignored:", len(raw.drop(columns=["Person ID"]).drop_duplicates()))

Many rows repeat identical values once the `Person ID` column is ignored. The effective number of
*independent* examples is therefore much smaller than the row count, which is why the training code keeps identical rows
together when splitting.

## 3. Cleaning (deterministic; nothing is invented)
Blood pressure text such as `126/83` becomes numeric systolic/diastolic columns; `Normal Weight` is merged into `Normal`.

In [ ]:
clean = clean_dataframe(raw)
roles = get_feature_roles(clean)
print("Numeric inputs    :", roles.numeric)
print("Categorical inputs:", roles.categorical)
print("Excluded          :", roles.excluded)
clean.head()

## 4. Descriptive statistics

In [ ]:
clean.describe().T

## 5. Figures
All figures are saved to `reports/figures/`.

In [ ]:
figures = eda.generate_eda_figures(clean, config.FIGURES_DIR)
print(len(figures), "figures written to", config.FIGURES_DIR)

In [ ]:
for key in ["target_distribution", "sleep_duration_distribution", "stress_distribution"]:
    print(eda.FIGURE_CAPTIONS[key][1]); display(Image(filename=str(figures[key])))

In [ ]:
for key in ["quality_vs_stress", "quality_vs_activity", "quality_vs_duration"]:
    print(eda.FIGURE_CAPTIONS[key][1]); display(Image(filename=str(figures[key])))

In [ ]:
display(Image(filename=str(figures["correlation_heatmap"])))

In [ ]:
for key in ["quality_by_bmi", "quality_by_gender", "quality_by_occupation", "quality_by_sleep_disorder"]:
    if key in figures:
        print(eda.FIGURE_CAPTIONS[key][1]); display(Image(filename=str(figures[key])))

## 6. Correlation of each numerical column with sleep quality

In [ ]:
num = [c for c in clean.columns if pd.api.types.is_numeric_dtype(clean[c]) and c != "Person ID"]
corr = clean[num].corr()[config.TARGET_COLUMN].drop(config.TARGET_COLUMN)
corr.sort_values(key=abs, ascending=False).round(3).to_frame("Pearson r with Quality of Sleep")

## 7. What to say (and not say)
* Stress level and sleep duration show the strongest associations with the recorded sleep-quality score in this file
  (see the table above for the exact values).
* These are **associations in one small dataset**, not evidence that changing stress or sleep duration changes sleep quality.
* `Sleep Disorder` is shown only as context; it is **not** used as a model input.
* The file does not document how it was collected; do not describe it as clinical or verified real-world data.